Week 15 · Day 6 — Reranking & Chunking
Why this matters

Even if embeddings + FAISS return results, they’re not always the best-quality context. Two tricks improve RAG:

Chunking: break long documents into smaller pieces so retrieval is precise.

Reranking: reorder retrieved candidates with a stronger cross-encoder model.

This helps the LLM get relevant, focused context.

Theory Essentials

Chunking: split docs into passages (e.g. 200–500 tokens). Avoids “long irrelevant block” problem.

Sliding window: overlap chunks so important phrases aren’t cut.

Reranking: second-stage model (cross-encoder) refines similarity after first-stage retrieval.

Trade-off: better precision but more compute.

Impact: fewer hallucinations, more accurate answers.

In [1]:
# Setup
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

# Docs (simulate longer ones)
docs = [
    "Paris is the capital of France. The Eiffel Tower is located in Paris and is a famous landmark.",
    "Rome is the capital of Italy. The Colosseum is an ancient amphitheater located in Rome.",
    "Madrid is the capital of Spain. The Prado Museum in Madrid is world-famous.",
]

# ---- Step 1: Chunking ----
def chunk_doc(doc, size=10):
    words = doc.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

chunked_docs = []
for d in docs:
    chunked_docs.extend(chunk_doc(d, size=8))  # smaller chunks

# ---- Step 2: Build index on chunks ----
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunked_docs).astype("float32")
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# ---- Step 3: First-stage retrieval ----
def retrieve(query, k=4):
    q_emb = embedder.encode([query]).astype("float32")
    D, I = index.search(q_emb, k)
    return [(chunked_docs[i], float(D[0][j])) for j, i in enumerate(I[0])]

# ---- Step 4: Reranking with CrossEncoder ----
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=2):
    pairs = [[query, doc] for doc in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Example
query = "Where is the Colosseum?"
first_stage = [doc for doc, _ in retrieve(query, k=4)]
print("First-stage:", first_stage)
print("Reranked:", rerank(query, first_stage, top_k=2))


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\AI-Mastery\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP d

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

First-stage: ['Rome is the capital of Italy. The Colosseum', 'is an ancient amphitheater located in Rome.', 'famous landmark.', 'Museum in Madrid is world-famous.']
Reranked: [('Rome is the capital of Italy. The Colosseum', np.float32(3.8181677)), ('is an ancient amphitheater located in Rome.', np.float32(-1.65535))]



## What the code does

1. **Chunking (Step 1)**

   * `chunk_doc` splits each long doc into **8-word chunks**.
   * Why: long passages blur meaning; small chunks make retrieval more targeted.

2. **Vector index on chunks (Step 2)**

   * `SentenceTransformer("all-MiniLM-L6-v2")` embeds each **chunk**.
   * FAISS `IndexFlatL2` stores those vectors and supports fast **nearest-neighbor** search by L2 distance.

3. **First-stage retrieval (Step 3)**

   * Your query “Where is the Colosseum?” is embedded and searched against the index (`k=4`).
   * Output is the **top-4 chunks** by L2 similarity (really: *smallest* distance).

4. **Reranking (Step 4)**

   * `CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")` reads **(query, chunk)** pairs and scores how relevant each chunk is.
   * Unlike bi-encoder embeddings, the cross-encoder **looks at both texts together**, so it’s much better at picking the *exact* match.
   * You sort by score and keep `top_k=2`.

## Interpreting your results

**First-stage** returned:

```
[
 'Rome is the capital of Italy. The Colosseum',
 'is an ancient amphitheater located in Rome.',
 'famous landmark.',
 'Museum in Madrid is world-famous.'
]
```

* The first two chunks are **correct** (they mention “Colosseum… Rome”).
* The other two are **noise**: “famous landmark.” (from the Paris doc tail) and a Madrid museum chunk.
  This happens because:

  * small chunks can be **too short/ambiguous**,
  * bi-encoder similarity is approximate and may pull semantically related but wrong snippets.

**Reranked** top-2:

```
('Rome is the capital of Italy. The Colosseum', 3.818)
('is an ancient amphitheater located in Rome.', -1.655)
```

* The cross-encoder ranks the **two Rome/Colosseum chunks highest** and pushes the Paris/Madrid noise down.
* Scores are **relevance logits** (not probabilities). Only **relative order** matters.

## What this shows (the lesson)

* **Bi-encoder retrieval = fast, good recall** → brings back some correct + some noise.
* **Cross-encoder rerank = slow, high precision** → promotes the truly relevant chunks to the top.



1) Core (10–15 min)

Task: Try the query “Where is the Prado Museum?” and see if reranking improves the top doc.

In [2]:
query = "Where is the Prado Museum?"
first_stage = [doc for doc, _ in retrieve(query, 4)]
print("Before:", first_stage)
print("After:", rerank(query, first_stage, top_k=1))


Before: ['Museum in Madrid is world-famous.', 'Madrid is the capital of Spain. The Prado', 'famous landmark.', 'is an ancient amphitheater located in Rome.']
After: [('Madrid is the capital of Spain. The Prado', np.float32(2.2782362))]


2) Practice (10–15 min)

Task: Change chunk size from 8 → 5 words. Rebuild the index and test again with “Eiffel Tower.”

In [3]:
chunked_docs = []
for d in docs:
    chunked_docs.extend(chunk_doc(d, size=5))
embeddings = embedder.encode(chunked_docs).astype("float32")
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(retrieve("Eiffel Tower", 3))


[('France. The Eiffel Tower is', 0.35860490798950195), ('a famous landmark.', 1.1307966709136963), ('located in Paris and is', 1.1641780138015747)]


3) Stretch (optional, 10–15 min)

Task: Compare results with vs without reranking for the query “Rome amphitheater.” Does reranker help surface Colosseum chunks?

In [4]:
query = "Rome amphitheater"
first_stage = [doc for doc, _ in retrieve(query, 4)]
print("Without rerank:", first_stage)
print("With rerank:", rerank(query, first_stage, top_k=2))


Without rerank: ['ancient amphitheater located in Rome.', 'Rome is the capital of', 'Italy. The Colosseum is an', 'a famous landmark.']
With rerank: [('ancient amphitheater located in Rome.', np.float32(7.2003055)), ('Rome is the capital of', np.float32(-2.0414073))]


Mini-Challenge (≤40 min)

Improve Context Quality with Chunking + Reranking

Take your doc set from Day 4.

Implement chunking with overlap (e.g., 50 tokens with 10 overlap).

Retrieve top-5 → rerank with CrossEncoder → keep top-2.

Test on at least 3 queries.

Acceptance Criteria:

Clear improvement in final retrieved context (more specific).

Chunking function with overlap implemented.

Reranker consistently ranks the truly relevant passage higher.

In [5]:
# Setup
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# ----- Step 1: Doc set (10 facts) -----
docs = [
    "The Eiffel Tower is located in Paris and was completed in 1889 for the World's Fair.",
    "The Colosseum in Rome is the largest ancient amphitheater ever built.",
    "The Prado Museum in Madrid holds one of the finest collections of European art.",
    "The Brandenburg Gate in Berlin is an 18th-century neoclassical monument.",
    "Big Ben is the nickname for the Great Bell of the clock at the Palace of Westminster in London.",
    "The Acropolis of Athens is an ancient citadel on a rocky outcrop above the city of Athens.",
    "The Leaning Tower of Pisa is known for its unintended tilt and is located in Pisa, Italy.",
    "The Statue of Liberty in New York was a gift from France and symbolizes freedom.",
    "The Great Wall of China stretches thousands of miles across northern China.",
    "The Taj Mahal in Agra, India, was commissioned in 1632 by Mughal emperor Shah Jahan."
]

# ----- Step 2: Chunking with overlap -----
def chunk_doc(doc, size=10, overlap=3):
    words = doc.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        chunk = " ".join(words[i:i+size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunked_docs = []
for d in docs:
    chunked_docs.extend(chunk_doc(d, size=10, overlap=3))

# ----- Step 3: Embedding + FAISS -----
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunked_docs).astype("float32")
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

def retrieve(query, k=5):
    q_emb = embedder.encode([query]).astype("float32")
    D, I = index.search(q_emb, k)
    return [chunked_docs[i] for i in I[0]]

# ----- Step 4: Rerank with CrossEncoder -----
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=2):
    pairs = [[query, c] for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# ----- Step 5: Demo on queries -----
queries = [
    "Where is the Colosseum located?",
    "What city has the Leaning Tower?",
    "Which monument in Berlin was built in the 18th century?"
]

for q in queries:
    first_stage = retrieve(q, k=5)
    reranked = rerank(q, first_stage, top_k=2)

    print(f"\nQuery: {q}")
    print(" First-stage retrieved:")
    for c in first_stage:
        print("  -", c)
    print(" Reranked (top-2):")
    for c, s in reranked:
        print(f"  - {c}  (score={s:.2f})")



Query: Where is the Colosseum located?
 First-stage retrieved:
  - The Colosseum in Rome is the largest ancient amphitheater ever
  - ancient amphitheater ever built.
  - city of Athens.
  - Palace of Westminster in London.
  - 18th-century neoclassical monument.
 Reranked (top-2):
  - The Colosseum in Rome is the largest ancient amphitheater ever  (score=7.44)
  - Palace of Westminster in London.  (score=-8.13)

Query: What city has the Leaning Tower?
 First-stage retrieved:
  - The Leaning Tower of Pisa is known for its unintended
  - citadel on a rocky outcrop above the city of Athens.
  - for its unintended tilt and is located in Pisa, Italy.
  - Palace of Westminster in London.
  - in Pisa, Italy.
 Reranked (top-2):
  - The Leaning Tower of Pisa is known for its unintended  (score=3.91)
  - for its unintended tilt and is located in Pisa, Italy.  (score=-2.46)

Query: Which monument in Berlin was built in the 18th century?
 First-stage retrieved:
  - The Brandenburg Gate in Berlin

Notes / Key Takeaways

Long docs must be chunked to avoid irrelevant retrieval.

Smaller chunks = more precise, but too small = fragmented context.

Reranking improves precision, especially with ambiguous queries.

Trade-off: higher compute cost vs better answers.

This is exactly what modern production RAG stacks do (retriever + reranker).

Reflection

In what cases would chunking hurt retrieval performance?

Why not always use reranking, if it improves results?

In what cases would chunking hurt retrieval performance?

If chunks are too small, they lose context and become ambiguous (“famous landmark” could match anything).

If chunks are misaligned, important information might be split across boundaries so no chunk alone is fully relevant.

Over-chunking also increases the index size, making retrieval noisier and less efficient.

Why not always use reranking, if it improves results?

Reranking with a cross-encoder is computationally expensive since it runs a full forward pass per candidate pair.

For large-scale systems with millions of docs, reranking every query would be too slow and costly.

In practice, we combine both: bi-encoder retrieval for speed, then rerank only a small top-N set for precision.